<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.3_preco_de_imoveis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.3 — Preço de imóveis: regressão com rede densa

Nos notebooks 5.1 e 5.2, a rede classificava. Aqui ela faz **regressão**, como no módulo 2 (prever o salário pelos anos de experiência), mas com uma rede densa e muitas features: prever o preço de venda de uma casa a partir das suas características (área, número de quartos e banheiros, ano de construção, localização etc.).

A saída da rede é um único número (o preço) e a função de custo volta a ser o erro quadrático médio (`nn.MSELoss`).

## Objetivos

- Adaptar o pipeline dos notebooks 5.1 e 5.2 para regressão: uma saída, sem ativação, e `nn.MSELoss`.
- Normalizar também o alvo (`MinMaxScaler`) e desfazer a normalização para interpretar o erro em dólares.
- Avaliar um modelo de regressão com MAE e RMSE e comparar com um baseline.

## Importação das bibliotecas

In [ ]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

torch.manual_seed(1111)   # reprodutibilidade: inicialização dos pesos, dropout e embaralhamento
np.random.seed(1111)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("device:", device)

## Dados

Usamos o [House Sales in King County, USA](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction), do Kaggle: 21.613 vendas de casas na região de Seattle entre maio de 2014 e maio de 2015, com o preço em dólares. O endereço de download da API do Kaggle funciona sem login (verificado em novembro de 2026); se falhar, baixe o zip pela página e coloque `kc_house_data.csv` em `data/`.

In [ ]:
# Baixa o zip do Kaggle (não exige login) para ./data e extrai o CSV.
os.makedirs("data", exist_ok=True)
url = "https://www.kaggle.com/api/v1/datasets/download/harlfoxem/housesalesprediction"
zip_path = "data/housesalesprediction.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")

df = pd.read_csv("data/kc_house_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["price"].describe()

Usamos 16 features numéricas. Ficam de fora `id` (identificador da venda), `date` (data da venda) e `sqft_living15`/`sqft_lot15` (áreas médias das 15 casas vizinhas). Todas as colunas usadas já são numéricas; `zipcode` (CEP) entra como número, uma simplificação: o CEP é uma categoria, e a localização também está em `lat` e `long`.

In [ ]:
features = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "waterfront", "view", "condition",
            "grade", "sqft_above", "sqft_basement", "yr_built", "yr_renovated", "zipcode", "lat", "long"]
target = "price"

X = df[features].values
Y = df[target].values
print("X:", X.shape, "| Y:", Y.shape)

### Treino, teste e normalização

As features são padronizadas com `StandardScaler`, como antes. O alvo também é normalizado: os preços vão de US\$ 75 mil a US\$ 7,7 milhões, e um alvo nessa escala geraria custos e gradientes enormes no início do treino. O `MinMaxScaler` leva o preço para o intervalo $[0, 1]$:

$$y_{\text{norm}} = \frac{y - y_{\min}}{y_{\max} - y_{\min}}$$

em que $y_{\min}$ e $y_{\max}$ são o menor e o maior preço **do treino**. Os dois scalers são ajustados só no treino. Como o custo é calculado sobre $y_{\text{norm}}$, os valores impressos durante o treino não estão em dólares; para interpretá-los, desfazemos a normalização com `y_scaler.inverse_transform` (exercício 1).

O conjunto de teste também escolhe a melhor época, então faz o papel de validação.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train)
X_test = X_scaler.transform(X_test)

y_scaler = MinMaxScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))   # reshape: o scaler espera shape (M, 1)
y_test = y_scaler.transform(y_test.reshape(-1, 1))
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("primeiros alvos normalizados:", y_train[:3].ravel())

Os alvos ficam com shape `(M, 1)` e tipo `float32`, o mesmo shape da saída da rede, `(N, 1)`, como pede o `nn.MSELoss`.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_dataset = TabularDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
test_dataset = TabularDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))
train_dataset[5]

In [ ]:
train_kwargs = {"batch_size": 100, "shuffle": True}    # embaralha o treino a cada época
test_kwargs = {"batch_size": 100, "shuffle": False}    # a ordem não importa na avaliação
if use_cuda:
    cuda_kwargs = {"num_workers": 1, "pin_memory": True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = DataLoader(train_dataset, **train_kwargs)
test_loader = DataLoader(test_dataset, **test_kwargs)
print("mini-batches por época:", len(train_loader))

## Modelo

A mesma rede dos notebooks anteriores (128 e 256 neurônios, ReLU, dropout 0,3), agora com **uma** saída e sem ativação nela: o preço normalizado pode assumir qualquer valor real.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(X_train.shape[1], 128)
        self.drop1 = nn.Dropout(0.3)
        self.hidden = nn.Linear(128, 256)
        self.drop2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.input(x)
        x = F.relu(x)
        x = self.drop1(x)
        x = self.hidden(x)
        x = F.relu(x)
        x = self.drop2(x)
        x = self.fc2(x)      # (N, 1): preço normalizado
        return x

model = Net()
print(model)
print("parâmetros:", sum(p.numel() for p in model.parameters()))

## Treinamento

Em regressão não há acurácia: `train` e `test` devolvem só o custo médio. A parada antecipada e o checkpoint passam a olhar o **menor custo de teste** em vez da maior acurácia. `StepLR` e parada antecipada foram explicados no notebook 5.1.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
    train_loss = running_loss / len(train_loader.dataset)
    return train_loss

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
    test_loss /= len(test_loader.dataset)
    return test_loss

In [ ]:
n_epochs = 100
lr = 0.001
lr_decay = 10       # a cada 10 épocas, a taxa de aprendizado cai pela metade
patience = 10       # parada antecipada após 10 épocas sem melhora no custo de teste
patience_counter = 0
best_loss = float("inf")
ckpt_path = "melhor_modelo_imoveis.pt"

history = {"train_loss": [], "test_loss": []}

torch.manual_seed(1111)
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()
scheduler = StepLR(optimizer, step_size=lr_decay, gamma=0.5)

print("Iniciando treinamento ...")
for epoch in range(1, n_epochs + 1):
    train_loss = train(model, device, train_loader, optimizer, criterion)
    test_loss = test(model, device, test_loader, criterion)
    scheduler.step()

    print(f"Época {epoch:3d} | custo treino: {train_loss:.6f} | custo teste: {test_loss:.6f}")
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)

    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), ckpt_path)
        print("    melhor época até agora: pesos salvos")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter == patience:
            print(f"Parada antecipada na época {epoch}")
            break

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_loss = test(model, device, test_loader, criterion)
print(f"Melhor modelo | custo de teste (MSE no preço normalizado): {test_loss:.6f}")

## Avaliação

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="treino")
plt.plot(range(1, len(history["test_loss"]) + 1), history["test_loss"], label="teste")
plt.yscale("log")
plt.title("Função de custo (MSE no preço normalizado)")
plt.xlabel("época")
plt.ylabel("custo (escala log)")
plt.legend()
plt.grid(True)
plt.show()

Nas primeiras épocas, o custo de treino fica acima do de teste porque é calculado com o dropout ligado e como média ao longo da época, enquanto os pesos ainda mudam muito; o de teste é calculado com o dropout desligado, ao fim da época. Depois, o custo de treino continua caindo e o de teste oscila: a parada antecipada guarda os pesos da época de menor custo de teste.

## Exercícios

**1.** Um custo da ordem de 0,0004 no preço normalizado não diz muito. Calcule as previsões do melhor modelo para o conjunto de teste, desfaça a normalização com `y_scaler.inverse_transform` e calcule, em dólares, o erro absoluto médio (MAE) e a raiz do erro quadrático médio (RMSE):

$$\text{MAE} = \frac{1}{M}\sum_{i=1}^{M} |\hat{y}_i - y_i| \qquad \text{RMSE} = \sqrt{\frac{1}{M}\sum_{i=1}^{M} (\hat{y}_i - y_i)^2}$$

em que $M$ é o número de casas de teste, $y_i$ o preço real e $\hat{y}_i$ o previsto. Compare com um baseline que prevê, para toda casa, o preço médio do treino.

In [ ]:
# TODO: escreva sua solução aqui
print(f"rede:     MAE = US$ {mae:,.0f} | RMSE = US$ {rmse:,.0f}")
print(f"baseline: MAE = US$ {mae_base:,.0f} | RMSE = US$ {rmse_base:,.0f}")

**2.** Faça um gráfico de dispersão do preço previsto contra o preço real das casas de teste, com a reta $\hat{y} = y$ como referência. Em que faixa de preço o modelo erra mais? Por que o RMSE é maior que o MAE?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- Para regressão, a rede tem uma saída sem ativação e usa `nn.MSELoss`; a parada antecipada passa a acompanhar o menor custo de teste.
- Normalizar o alvo deixa o treino estável, mas o custo deixa de estar em unidades do problema: desfaça a normalização para interpretar o erro.
- MAE e RMSE em dólares, comparados com um baseline (prever a média), dizem se o modelo é útil; o RMSE pesa mais os erros grandes.
- Regiões com poucos exemplos no treino (casas muito caras) concentram os maiores erros.